# LightRAG Test Notebook

This notebook tests the reusable LightRAG module in `modules/lightrag`.

## What it does
- Loads `LightRAGModule`
- Ingests documents from `data/user/ruling`
- Runs a sample query against the indexed data

> If needed, install dependencies first:
> `uv add lightrag-hku pypdf`

In [6]:
from functools import partial
from pathlib import Path
import importlib
import sys

import numpy as np
import os

print("Directory: ", os.getcwd())

sys.path.append("/Users/lscm/Project/PCS/HSCodeSearch")
from dotenv import load_dotenv
import os

load_dotenv()  # Loads .env from current directory (or parent dirs)



Directory:  /Users/lscm/Project/PCS/HSCodeSearch/modules/lightrag


True

In [ ]:
import os
import logging
from lightrag import LightRAG, QueryParam
from lightrag.llm.ollama import ollama_model_complete,ollama_embed  # Direct import
from lightrag.utils import EmbeddingFunc

logging.basicConfig(format="%(levelname)s:%(message)s", level=logging.INFO)

WORKING_DIR = "../../../data/knowledge_base/test2"
os.makedirs(WORKING_DIR, exist_ok=True)

rag = LightRAG(
        working_dir=WORKING_DIR,
        llm_model_func=ollama_model_complete,
        llm_model_kwargs={"host": "http://10.103.1.23:11434"},
        embedding_func=EmbeddingFunc(
            embedding_dim=1024, # Match embed model
            max_token_size=8192,
            func=ollama_embed,
            model_name="qwen3-embedding:4b", # Or your pulled model,
        ),
    )


folder_path = "../../../data/data_sources/user/test/"  # path to folder with PDFs
texts = []

for filename in os.listdir(folder_path):
    if filename.lower().endswith('.pdf'):
        file_path = os.path.join(folder_path, filename)
        try:
            text_content = textract.process(file_path)
            texts.append(text_content.decode('utf-8'))
            print(f"Ingested {filename}")
        except Exception as e:
            print(f"Failed {filename}: {e}")

# Batch insert all extracted texts
rag.insert(texts)

INFO: [] Created new empty graph file: ../../../data/knowledge_base/test2/graph_chunk_entity_relation.graphml
INFO:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': '../../../data/knowledge_base/test2/vdb_entities.json'} 0 data
INFO:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': '../../../data/knowledge_base/test2/vdb_relationships.json'} 0 data
INFO:Init {'embedding_dim': 1024, 'metric': 'cosine', 'storage_file': '../../../data/knowledge_base/test2/vdb_chunks.json'} 0 data


FileNotFoundError: [Errno 2] No such file or directory: '../../../data/user/test/'

In [ ]:
question = "What kinds of product classification rulings are in this dataset?"
answer = rag.query(question, mode="hybrid")
answer

In [ ]:
# Optional: run this when you are done to close storage resources.
rag.close()